# Gaussian Wells + SARF Anchors on Fock-PARFLM v2.1 — TinyStories Ablation

## Motivation

The SQ3 (log-sum-exp mixture) structured V_θ trained successfully on TinyStories
(10.36 PPL, A2 winner config) but exhibits **persistent training instabilities**
on OpenWebText due to the structurally unbounded potential: V_θ → +∞ as h escapes
all well centres, producing unbounded gradients and repeated watchdog triggers.

This notebook evaluates a **principled architectural fix**: replacing the SQ3
log-sum-exp V_θ with **Gaussian mixture-PDF wells** (bounded V ∈ [-Σw_k, 0]) and
optionally centring them on **SARF anchors** (PMI-peak token embeddings for global
semantic coverage). The TinyStories ablation establishes the PPL baseline before
committing to an OpenWebText-scale retrain.

## Experiment cells

| Cell | V_θ variant | Key property | Why interesting |
|------|-----------|--------------|----------------|
| `G1` | Gaussian K=8 (learned centres) | Bounded V, same K as SQ3 A2 | Direct SQ3 replacement, tests expressivity |
| `G2` | Gaussian K=16 (learned centres) | Higher capacity bounded | Does more wells help? |
| `G3` | SARF N_S=64 (frozen PMI anchors) | Global coverage, tiny param count | Full SARF architectural proposal |
| `G4` | SARF N_S=128 (frozen PMI anchors) | Denser anchor coverage | Does more anchors help? |
| `G5` | Gaussian K=8 + background quad | Bounded + mild global restoring | Hybrid: Gaussian safety + quadratic pull |
| `S1` | SQ3 K=8 (log-sum-exp, baseline) | Unbounded, proven 10.36 PPL | Reference — the A2 winner |
| `M1` | MLP baseline (v_hidden=1024) | Reference MLP | Reproduce ~8.95 PPL |

**Primary comparison:** G1 (Gaussian K=8) and G3 (SARF N_S=64) vs S1 (SQ3 K=8)
and M1 (MLP baseline).

## Architecture

All cells use `FockMultiXiPARFLM` with d=256, L=8, fixed_gamma=0.3,
logfreq mass, xi_alpha_inits=[0.25, 0.50, 0.75, 0.95].
PARFLM: structural_competitive routing, top_k=8, Gumbel gates.
Fock v2.1: 16 registers, stack discipline, reverse channel.
Only V_θ is swapped — all other dynamics are identical.

## Key structural properties

| Property | SQ3 (S1) | Gaussian (G1/G2) | SARF (G3/G4) |
|----------|----------|------------------|---------------|
| V range | (-∞, +∞) | [-Σw_k, 0] | [-Σw_j, 0] |
| Force bound | unbounded | 0.607 w/σ | 0.607 w/σ |
| V² penalty | unbounded | bounded | bounded |
| Well centres | learned μ_k(ξ) | learned μ_k(ξ) | frozen PMI anchors |
| Escape risk | none (global pull) | possible (no far-field force) | none (anchor coverage) |
| V_θ params | ~25k | ~25k | ~128 + frozen |

## 0. Environment setup + cell selector

In [ ]:
CELL = 'G1'       # one of: G1..G5, S1, M1
SEED = 0

REPO_URL        = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH     = 'main'
COLAB_REPO_PATH = '/content/semsimula-paper'
GDRIVE_OUT_REL  = 'semsimula_fock_gaussian_sarf_vtheta'

import os, sys, shutil, subprocess, json, time, math
from pathlib import Path

os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd: str) -> None:
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_OUT = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'GDrive output root = {GDRIVE_OUT}')

    REPO_ROOT = Path(COLAB_REPO_PATH)
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(
            f'git clone --depth 1 --branch {REPO_BRANCH} '
            f'{REPO_URL} {REPO_ROOT}'
        )
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: could not refresh repo ({e}); using existing checkout.')

    DATA_CACHE = GDRIVE_OUT / 'data'
    DATA_CACHE.mkdir(exist_ok=True)
    repo_data_dir = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data_dir.is_symlink():
        repo_data_dir.unlink()
    elif repo_data_dir.is_dir():
        shutil.rmtree(repo_data_dir)
    repo_data_dir.symlink_to(DATA_CACHE)
    print(f'data/ -> {DATA_CACHE}')

    _sh('pip install -q transformers huggingface_hub pyarrow')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    GDRIVE_OUT = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / 'fock_gaussian_sarf_vtheta'
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)

SARF_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
for sub in ['', 'multixi', 'parf', 'sarf_mass_variant', 'energetic_minima', 'scaleup']:
    d = str(SARF_DIR / sub) if sub else str(SARF_DIR)
    if d not in sys.path:
        sys.path.insert(0, d)

RESULTS_ROOT = GDRIVE_OUT
RUN_DIR = RESULTS_ROOT / CELL / f'seed{SEED}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f'Run output dir = {RUN_DIR}')

## 1. GPU check

In [ ]:
import torch
import torch.nn as nn
import numpy as np

if torch.cuda.is_available():
    device = 'cuda'
    props = torch.cuda.get_device_properties(0)
    total_memory = props.total_memory / 1e9
    print(f'GPU: {props.name}  ({total_memory:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    device = 'cpu'
    total_memory = 0
    print('WARNING: no GPU detected — training will be very slow')

print(f'device = {device}')

## 2. Experiment recipes

In [ ]:
RECIPES = {
    # ─── Gaussian wells (learned centres) ─────────────────────────
    'G1': {
        'desc': 'Gaussian K=8 (learned centres, bounded V)',
        'v_theta_kind': 'gaussian', 'K_mix': 8, 'w_scale': 1.0,
        'sarf_n_anchors': None,
        'bg_quad_eps': 0.0,
        'xi_channels': 4, 'lambda_v': 1e-2,
    },
    'G2': {
        'desc': 'Gaussian K=16 (learned centres, higher capacity)',
        'v_theta_kind': 'gaussian', 'K_mix': 16, 'w_scale': 1.0,
        'sarf_n_anchors': None,
        'bg_quad_eps': 0.0,
        'xi_channels': 4, 'lambda_v': 1e-2,
    },
    # ─── SARF anchors (frozen PMI-peak centres) ───────────────────
    'G3': {
        'desc': 'SARF N_S=64 Gaussian wells (frozen PMI anchors)',
        'v_theta_kind': 'sarf', 'K_mix': None, 'w_scale': 1.0,
        'sarf_n_anchors': 64,
        'bg_quad_eps': 0.0,
        'xi_channels': 4, 'lambda_v': 1e-2,
    },
    'G4': {
        'desc': 'SARF N_S=128 Gaussian wells (denser anchor coverage)',
        'v_theta_kind': 'sarf', 'K_mix': None, 'w_scale': 1.0,
        'sarf_n_anchors': 128,
        'bg_quad_eps': 0.0,
        'xi_channels': 4, 'lambda_v': 1e-2,
    },
    # ─── Gaussian + background quadratic ──────────────────────────
    'G5': {
        'desc': 'Gaussian K=8 + eps*||h||^2 background (hybrid)',
        'v_theta_kind': 'gaussian', 'K_mix': 8, 'w_scale': 1.0,
        'sarf_n_anchors': None,
        'bg_quad_eps': 1e-4,
        'xi_channels': 4, 'lambda_v': 1e-2,
    },
    # ─── SQ3 baseline (unbounded reference) ──────────────────────
    'S1': {
        'desc': 'SQ3 K=8 log-sum-exp (A2 winner, unbounded reference)',
        'v_theta_kind': 'sq3', 'K_mix': 8, 'w_scale': None,
        'sarf_n_anchors': None,
        'bg_quad_eps': 0.0,
        'xi_channels': 4, 'lambda_v': 1e-2,
    },
    # ─── MLP baseline ─────────────────────────────────────────────
    'M1': {
        'desc': 'MLP baseline v_hidden=1024 (~8.95 PPL reference)',
        'v_theta_kind': 'mlp', 'K_mix': None, 'w_scale': None,
        'sarf_n_anchors': None,
        'bg_quad_eps': 0.0,
        'xi_channels': 4, 'lambda_v': 0.0,
    },
}

if CELL not in RECIPES:
    raise ValueError(f'CELL must be one of {sorted(RECIPES)}; got {CELL!r}')

recipe = RECIPES[CELL]

# ─── Shared architecture ──────────────────────────────────────────
D              = 256
L              = 8
V_HIDDEN       = 1024
V_DEPTH        = 3
VOCAB_SIZE     = 50257
MAX_LEN        = 1024
DT             = 1.0
FIXED_GAMMA    = 0.30
XI_CHANNELS    = recipe['xi_channels']
XI_LEARNABLE   = True
LAMBDA_V       = recipe['lambda_v']
BG_QUAD_EPS    = recipe['bg_quad_eps']

STEPS          = 16000
BATCH          = 16
GRAD_ACCUM     = 1
BLOCK          = 512
LR             = 5e-4
WD             = 0.01
WARMUP         = 400
GRAD_CLIP      = 1.0
EVAL_INTERVAL  = 400
EVAL_ITERS     = 40
LOG_INTERVAL   = 50

XI_ALPHA_INITS = [0.25, 0.50, 0.75, 0.95]

print(f'Cell {CELL}: {recipe["desc"]}')
print(f'  V_theta = {recipe["v_theta_kind"]}')
if recipe['K_mix'] is not None:
    print(f'  K_mix = {recipe["K_mix"]}')
if recipe['sarf_n_anchors'] is not None:
    print(f'  SARF N_S = {recipe["sarf_n_anchors"]}')
if recipe['bg_quad_eps'] > 0:
    print(f'  background quadratic eps = {recipe["bg_quad_eps"]}')
print(f'  xi_channels = {XI_CHANNELS}')
print(f'  lambda_V = {LAMBDA_V}')
print(f'  steps={STEPS}  batch={BATCH}  block={BLOCK}  d={D}  L={L}')

## 3. Progress review (no GPU needed)

In [ ]:
import json, math
from pathlib import Path

_run_dir = RUN_DIR
_log_path = _run_dir / 'training_log.jsonl'

print('─' * 55)
print(f'PROGRESS REVIEW  —  Cell {CELL}: {recipe["desc"]}')
print('─' * 55)

_best_ckpt = _run_dir / f'ckpt_best.pt'
if _best_ckpt.exists():
    try:
        _bd = torch.load(_best_ckpt, map_location='cpu', weights_only=False)
        print(f'\nBest checkpoint: PPL {_bd.get("val_ppl", float("nan")):.2f}'
              f'  at step {_bd.get("step", 0):,}')
    except Exception:
        pass

_latest_ckpt = _run_dir / f'ckpt_latest.pt'
if _latest_ckpt.exists():
    try:
        _ld = torch.load(_latest_ckpt, map_location='cpu', weights_only=False)
        print(f'Latest checkpoint: step {_ld.get("step", 0):,}'
              f'  PPL {_ld.get("val_ppl", float("nan")):.2f}')
    except Exception:
        pass

eval_entries = []
if _log_path.exists():
    with open(_log_path) as f:
        for line in f:
            try:
                e = json.loads(line)
                if 'val_ppl' in e:
                    eval_entries.append((e['step'], e['val_ppl']))
            except Exception:
                pass
if eval_entries:
    print(f'\nVal PPL history ({len(eval_entries)} evals):')
    best_s, best_p = min(eval_entries, key=lambda x: x[1])
    last_s, last_p = eval_entries[-1]
    for step, ppl in eval_entries[-10:]:
        marker = ' ← best' if (step, ppl) == (best_s, best_p) else ''
        print(f'  step {step:>6,}:  PPL {ppl:>8.2f}{marker}')
    print(f'\n  Best PPL : {best_p:.2f}  at step {best_s:,}')
    print(f'  Latest   : {last_p:.2f}  at step {last_s:,}')
    print(f'  Progress : {last_s:,} / {STEPS:,} ({100*last_s/STEPS:.1f}%)')
else:
    print('\n  No training data yet.')

print('\n' + '─' * 55)

## 4. Load TinyStories

In [ ]:
from data_module import load_tiny_stories, get_batch

train_ids, val_ids = load_tiny_stories(max_train_tokens=5_000_000)
print(f'train: {len(train_ids):,} tokens   val: {len(val_ids):,} tokens')

rng = np.random.default_rng(SEED)

## 5. Build model + V_θ swap

Build the base Fock-PARFLM v2.1 model, then swap V_θ according to the
selected recipe.

For SARF cells (G3/G4), we compute a PMI matrix from the training tokens
and select the top-N_S PMI-peak token embeddings as frozen anchor positions.

In [ ]:
from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig
import model_fock_parf_v2
import model_parf_multixi
import model_parf
import model_parf_sparse
import torch.nn.functional as F_torch

# ── Logfreq surprisal ─────────────────────────────────────────────
LOGFREQ_PATH = SARF_DIR / 'scaleup' / 'results' / 'logfreq_surprisal_tinystories.npy'
DRIVE_LOGFREQ = RESULTS_ROOT / 'logfreq_surprisal_tinystories.npy'

if LOGFREQ_PATH.exists():
    LOGFREQ_FILE = LOGFREQ_PATH
    print(f'Using bundled logfreq: {LOGFREQ_FILE}')
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_FILE = DRIVE_LOGFREQ
    print(f'Using Drive-cached logfreq: {LOGFREQ_FILE}')
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_FILE = DRIVE_LOGFREQ
    LOGFREQ_FILE.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_FILE, surprisal)
    print(f'Built logfreq from train_ids; saved to {LOGFREQ_FILE}')

# ── Build model ───────────────────────────────────────────────────
torch.manual_seed(SEED)

cfg = FockMultiXiPARFConfig(
    vocab_size=VOCAB_SIZE, d=D, max_len=MAX_LEN,
    L=L, v_hidden=V_HIDDEN, v_depth=V_DEPTH, dt=DT,
    mass_mode='logfreq',
    logfreq_path=str(LOGFREQ_FILE),
    logfreq_init_alpha=0.1,
    init_gamma=1.0,
    fixed_gamma=FIXED_GAMMA,
    causal_force=True,
    ln_after_step=True,
    xi_channels=XI_CHANNELS,
    xi_alpha_inits=XI_ALPHA_INITS,
    xi_learnable=XI_LEARNABLE,
    xi_alpha_init_mode='explicit',
    v_phi_kind='structural_competitive',
    v_phi_phi_hidden=128,
    v_phi_theta_hidden=128,
    top_k=8,
    score_head_hidden=32,
    gumbel_tau_init=1.0,
    gumbel_tau_min=0.3,
    gumbel_noise=True,
    use_gathered_v_phi=True,
    use_layer_checkpoint=True,
    ln_before_distance=True,
    per_layer_v_phi_scale=True,
    fock_version='v2',
    n_registers=16,
    register_salience_decay=0.5,
    register_salience_threshold=0.005,
    creation_gate_hidden=64,
    stack_discipline=True,
    d_k=64,
    tau_create_init=8.0,
    reverse_channel=True,
    per_register_tau=True,
    per_register_keys=True,
    ortho_register_init=True,
)
model = FockMultiXiPARFLM(cfg).to(device)

n_total_before = sum(p.numel() for p in model.parameters())
n_v_theta_before = sum(p.numel() for p in model.V_theta.parameters())
print(f'Before swap: total={n_total_before:,}  V_theta={n_v_theta_before:,}')

# ── V_theta swap ──────────────────────────────────────────────────
vkind = recipe['v_theta_kind']
xi_d = XI_CHANNELS * D

if vkind == 'gaussian':
    from model_gaussian_vtheta import MixtureGaussianVTheta, GaussianVThetaMultiXiAdapter
    inner = MixtureGaussianVTheta(
        d=D, K=recipe['K_mix'], w_scale=recipe['w_scale'], xi_d=xi_d,
    )
    model.V_theta = GaussianVThetaMultiXiAdapter(inner, K=XI_CHANNELS, d=D).to(device)
    print(f'[{CELL}] V_theta -> Gaussian(K={recipe["K_mix"]}, w_scale={recipe["w_scale"]})')

elif vkind == 'sarf':
    from model_gaussian_vtheta import SARFGaussianVTheta, GaussianVThetaMultiXiAdapter
    N_S = recipe['sarf_n_anchors']
    print(f'[{CELL}] Computing SARF anchors (N_S={N_S}) from TinyStories PMI ...')

    # PMI computation from training token co-occurrences
    WINDOW = 5
    TOP_V = 8192  # restrict to top-8k most frequent tokens for PMI
    token_counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE)
    top_v_ids = np.argsort(-token_counts)[:TOP_V]
    id_to_local = np.full(VOCAB_SIZE, -1, dtype=np.int64)
    id_to_local[top_v_ids] = np.arange(TOP_V)

    cooc = np.zeros((TOP_V, TOP_V), dtype=np.float64)
    local_ids = id_to_local[train_ids.astype(np.int64)]
    for offset in range(1, WINDOW + 1):
        a = local_ids[:-offset]
        b = local_ids[offset:]
        valid = (a >= 0) & (b >= 0)
        np.add.at(cooc, (a[valid], b[valid]), 1.0)
    cooc = cooc + cooc.T

    row_sums = cooc.sum(axis=1, keepdims=True)
    total = cooc.sum()
    expected = row_sums * row_sums.T / total
    with np.errstate(divide='ignore', invalid='ignore'):
        pmi = np.log(cooc / np.maximum(expected, 1e-12))
    pmi = np.nan_to_num(pmi, nan=0.0, posinf=0.0, neginf=-20.0)

    # Select top-N_S PMI-peak tokens
    np.fill_diagonal(pmi, -np.inf)
    pmi_peaks = pmi.max(axis=1)
    anchor_local_ids = np.argsort(-pmi_peaks)[:N_S]
    anchor_token_ids = top_v_ids[anchor_local_ids]

    anchor_positions = model.E.weight.data[anchor_token_ids].detach().clone()
    print(f'  SARF anchors: {N_S} tokens selected')
    print(f'  PMI peak range: [{pmi_peaks[anchor_local_ids[-1]]:.2f}, '
          f'{pmi_peaks[anchor_local_ids[0]]:.2f}]')

    inner = SARFGaussianVTheta(
        d=D, anchor_positions=anchor_positions, xi_d=xi_d,
        w_scale=recipe['w_scale'],
    )
    model.V_theta = GaussianVThetaMultiXiAdapter(inner, K=XI_CHANNELS, d=D).to(device)
    print(f'[{CELL}] V_theta -> SARF Gaussian(N_S={N_S}, frozen anchors)')

    sarf_meta = {
        'anchor_token_ids': anchor_token_ids.tolist(),
        'pmi_peaks': pmi_peaks[anchor_local_ids].tolist(),
        'n_anchors': N_S, 'window': WINDOW, 'top_v': TOP_V,
    }
    with open(RUN_DIR / f'sarf_anchors_{CELL}.json', 'w') as f:
        json.dump(sarf_meta, f, indent=2)

elif vkind == 'sq3':
    from model_structured_vtheta import MixtureQuadraticVTheta
    from model_structured_vtheta_multixi import StructuredVThetaMultiXiAdapter
    inner = MixtureQuadraticVTheta.__new__(MixtureQuadraticVTheta)
    nn.Module.__init__(inner)
    inner.d = D
    inner.K = recipe['K_mix']
    inner.tau = 1.0
    inner.mu_proj = nn.Linear(xi_d, recipe['K_mix'] * D)
    inner.a_proj  = nn.Linear(xi_d, recipe['K_mix'] * D)
    inner.pi_proj = nn.Linear(xi_d, recipe['K_mix'])
    inner.b_proj  = nn.Linear(xi_d, 1)
    inner._init_weights(0.0)
    model.V_theta = StructuredVThetaMultiXiAdapter(inner, K=XI_CHANNELS, d=D).to(device)
    print(f'[{CELL}] V_theta -> SQ3 Mixture(K={recipe["K_mix"]})')

else:
    print(f'[{CELL}] Keeping MLP V_theta (v_hidden={V_HIDDEN}, v_depth={V_DEPTH})')

# ── Report ─────────────────────────────────────────────────────────
n_total = sum(p.numel() for p in model.parameters())
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
n_v_theta_buf = sum(b.numel() for b in model.V_theta.buffers())
IS_STRUCTURED = vkind != 'mlp'
IS_GAUSSIAN = vkind in ('gaussian', 'sarf')

print(f'\nAfter swap:')
print(f'  total params   = {n_total:,}')
print(f'  V_theta params = {n_v_theta:,}  ({n_v_theta/n_total*100:.1f}%)')
if n_v_theta_buf > 0:
    print(f'  V_theta buffers = {n_v_theta_buf:,} (frozen anchor positions)')
print(f'  V_theta reduction vs MLP: {n_v_theta_before:,} -> {n_v_theta:,} '
      f'({n_v_theta_before/max(n_v_theta,1):.0f}x)')
print(f'  xi_alpha init: {model.xi_alpha_values()}')
print(f'  Bounded V: {IS_GAUSSIAN}  (SQ3={not IS_GAUSSIAN and IS_STRUCTURED})')

## 6. Training loop

In [ ]:
def lr_at(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    progress = (step - WARMUP) / max(STEPS - WARMUP, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def forward_with_vreg(model, x, targets, lambda_v):
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
    logits = h_L @ model.E.weight.T
    loss_ntp = F_torch.cross_entropy(
        logits.reshape(-1, cfg.vocab_size),
        targets.reshape(-1),
    )

    v_reg_value = torch.tensor(0.0, device=x.device)
    if lambda_v > 0 and IS_STRUCTURED:
        xis = model.xi_module(h_L.detach())
        V_vals = model.V_theta(xis, h_L)
        if IS_GAUSSIAN:
            # Gaussian V is already bounded; plain V^2 is safe
            v_reg_value = (V_vals ** 2).mean()
        else:
            # SQ3: use log1p to prevent unbounded penalty
            v_reg_value = torch.log1p(V_vals ** 2).mean()
        # Optional background quadratic for escape prevention
        if BG_QUAD_EPS > 0:
            bg = BG_QUAD_EPS * (h_L ** 2).sum(dim=-1, keepdim=True).mean()
            loss = loss_ntp + lambda_v * v_reg_value + bg
        else:
            loss = loss_ntp + lambda_v * v_reg_value
    else:
        loss = loss_ntp

    return logits, loss, loss_ntp, v_reg_value


@torch.no_grad()
def evaluate_model():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(device)
        y = torch.from_numpy(yb).to(device)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


resume_step = 0
_latest_ckpt_path = RUN_DIR / 'ckpt_latest.pt'
_best_ckpt_path = RUN_DIR / 'ckpt_best.pt'

opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, betas=(0.9, 0.95), weight_decay=WD,
)

if _latest_ckpt_path.exists():
    ckpt = torch.load(_latest_ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    if 'optimizer_state_dict' in ckpt:
        try:
            opt.load_state_dict(ckpt['optimizer_state_dict'])
        except (ValueError, KeyError):
            print('[info] Optimizer state incompatible, starting fresh.')
    resume_step = ckpt.get('step', 0)
    print(f'Resumed from step {resume_step:,}  '
          f'(PPL {ckpt.get("val_ppl", "?")})  [{_latest_ckpt_path.name}]')

best_val_ppl = float('inf')
if _best_ckpt_path.exists():
    try:
        _bd = torch.load(_best_ckpt_path, map_location='cpu', weights_only=False)
        best_val_ppl = _bd.get('val_ppl', float('inf'))
        print(f'Restored best PPL from previous session: {best_val_ppl:.2f}')
        del _bd
    except Exception as e:
        print(f'[warn] Could not read best checkpoint: {e}')

log_path = RUN_DIR / 'training_log.jsonl'
log_f = log_path.open('a')

model.train()
t0 = time.time()
t_session = time.time()
val_ppl = float('nan')

for step in range(resume_step, STEPS):
    for g in opt.param_groups:
        g['lr'] = lr_at(step)

    opt.zero_grad(set_to_none=True)
    step_loss_ntp = 0.0
    step_v_reg = 0.0
    step_loss_total = 0.0

    for _acc in range(GRAD_ACCUM):
        xb, yb = get_batch(train_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(device)
        y = torch.from_numpy(yb).to(device)
        _, loss, loss_ntp, v_reg = forward_with_vreg(model, x, y, LAMBDA_V)
        (loss / GRAD_ACCUM).backward()
        step_loss_ntp   += loss_ntp.item() / GRAD_ACCUM
        step_v_reg      += v_reg.item()    / GRAD_ACCUM
        step_loss_total += loss.item()     / GRAD_ACCUM

    grad_norm = torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], GRAD_CLIP,
    ).item()
    opt.step()

    if (step + 1) % LOG_INTERVAL == 0 or step == 0:
        elapsed = time.time() - t_session
        steps_done = step + 1 - resume_step
        sec_per_step = elapsed / max(steps_done, 1)
        remaining = (STEPS - step - 1) * sec_per_step
        alphas_str = ','.join(f'{a:.3f}' for a in model.xi_alpha_values())
        print(f'[{CELL}] step {step+1:>5}/{STEPS}  '
              f'ntp={step_loss_ntp:.4f}  v_reg={step_v_reg:.4f}  '
              f'lr={lr_at(step):.2e}  grad={grad_norm:.3f}  '
              f'gamma={model.gamma.item():.3f}  '
              f'alpha=[{alphas_str}]  '
              f'{elapsed:.0f}s (~{remaining/60:.1f}m remaining)')
        log_entry = {
            'step': step + 1, 'train_loss': step_loss_ntp,
            'v_reg': step_v_reg, 'total_loss': step_loss_total,
            'lr': lr_at(step), 'grad_norm': grad_norm,
            'gamma': model.gamma.item(),
            'xi_alphas': model.xi_alpha_values(),
        }
        log_f.write(json.dumps(log_entry) + '\n')
        log_f.flush()

    if (step + 1) % EVAL_INTERVAL == 0 or (step + 1) == STEPS:
        val_loss = evaluate_model()
        val_ppl = math.exp(val_loss)
        is_best = val_ppl < best_val_ppl
        if is_best:
            best_val_ppl = val_ppl
        best_marker = '  *** NEW BEST ***' if is_best else ''
        print(f'  >> val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}  '
              f'best={best_val_ppl:.2f}{best_marker}')

        eval_entry = {
            'step': step + 1, 'val_loss': val_loss,
            'val_ppl': val_ppl, 'best_ppl': best_val_ppl,
        }
        log_f.write(json.dumps(eval_entry) + '\n')
        log_f.flush()

        if is_best:
            _best_state = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': opt.state_dict(),
                'step': step + 1, 'val_loss': val_loss,
                'val_ppl': val_ppl, 'gamma': model.gamma.item(),
                'xi_alphas': model.xi_alpha_values(),
                'cell': CELL, 'recipe': recipe, 'seed': SEED,
            }
            torch.save(_best_state, _best_ckpt_path)

    if (step + 1) % 4000 == 0 or (step + 1) == STEPS:
        _ckpt = {
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': opt.state_dict(),
            'step': step + 1,
            'val_ppl': val_ppl if (step+1) % EVAL_INTERVAL == 0 else best_val_ppl,
            'best_val_ppl': best_val_ppl,
            'gamma': model.gamma.item(),
            'xi_alphas': model.xi_alpha_values(),
            'cell': CELL, 'recipe': recipe, 'seed': SEED,
        }
        torch.save(_ckpt, _latest_ckpt_path)
        print(f'  [ckpt] saved {_latest_ckpt_path.name} at step {step+1}')

log_f.close()
print(f'\n[{CELL}] Training done.  total wall = {time.time()-t_session:.0f}s  '
      f'final_ppl = {val_ppl:.2f}  best_ppl = {best_val_ppl:.2f}')

## 7. Training curve

In [ ]:
import matplotlib.pyplot as plt

eval_entries = []
if log_path.exists():
    with open(log_path) as f:
        for line in f:
            try:
                e = json.loads(line)
                if 'val_ppl' in e:
                    eval_entries.append(e)
            except Exception:
                pass

if eval_entries:
    steps_arr = [e['step'] for e in eval_entries]
    ppls = [e['val_ppl'] for e in eval_entries]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(steps_arr, ppls, 'o-', label=f'{CELL}: {recipe["desc"]}', linewidth=1.5)
    ax.axhline(y=8.95, color='red', linestyle='--', alpha=0.7,
               label='MLP baseline (8.95 PPL)')
    ax.axhline(y=10.36, color='blue', linestyle='--', alpha=0.7,
               label='SQ3 K=8 (10.36 PPL)')
    ax.set_xlabel('Step')
    ax.set_ylabel('Val PPL')
    ax.set_title(f'Gaussian/SARF V_θ on Fock-PARFLM v2.1 — {CELL}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(RUN_DIR / f'training_curve_{CELL}.png', dpi=150)
    plt.show()
    print(f'Saved: {RUN_DIR / f"training_curve_{CELL}.png"}')
else:
    print('No eval data to plot.')

## 8. V_θ landscape + boundedness diagnostics

In [ ]:
v_samples = []
grad_samples = []
model.eval()
for _ in range(10):
    xb, _ = get_batch(val_ids, BATCH, BLOCK, rng)
    x = torch.from_numpy(xb).to(device)
    with torch.enable_grad():
        h0 = model._embed(x)
        h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
        xis = model.xi_module(h_L.detach())
        h_in = h_L.requires_grad_(True) if not h_L.requires_grad else h_L
        V_vals = model.V_theta(xis, h_in)
        v_samples.append(V_vals.detach().cpu().numpy().ravel())
        # Measure force magnitude
        grad_V = torch.autograd.grad(
            V_vals.sum(), h_in, create_graph=False, retain_graph=False,
        )[0]
        force_norms = grad_V.detach().norm(dim=-1).cpu().numpy().ravel()
        grad_samples.append(force_norms)

v_all = np.concatenate(v_samples)
f_all = np.concatenate(grad_samples)

ls = {
    'V_mean': float(v_all.mean()), 'V_std': float(v_all.std()),
    'V_min': float(v_all.min()), 'V_max': float(v_all.max()),
    'V_range': float(v_all.max() - v_all.min()),
    'force_mean': float(f_all.mean()), 'force_max': float(f_all.max()),
    'force_p99': float(np.percentile(f_all, 99)),
    'bounded': bool(IS_GAUSSIAN),
}
print(f'V_theta landscape stats ({"BOUNDED" if IS_GAUSSIAN else "UNBOUNDED"}):')
for k, v in ls.items():
    if isinstance(v, float):
        print(f'  {k:12s}: {v:.6f}')
    else:
        print(f'  {k:12s}: {v}')

with open(RUN_DIR / f'landscape_stats_{CELL}.json', 'w') as f:
    json.dump(ls, f, indent=2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(v_all, bins=80, edgecolor='none', alpha=0.8)
axes[0].axvline(ls['V_mean'], color='red', linestyle='--', alpha=0.6,
                label=f'mean={ls["V_mean"]:.4f}')
axes[0].set_xlabel('V_θ(ξ, h)')
axes[0].set_ylabel('count')
axes[0].set_title(f'{CELL} V_θ distribution')
axes[0].legend()

axes[1].hist(f_all, bins=80, edgecolor='none', alpha=0.8, color='orange')
axes[1].axvline(ls['force_p99'], color='red', linestyle='--', alpha=0.6,
                label=f'p99={ls["force_p99"]:.4f}')
axes[1].set_xlabel('||∇_h V_θ||')
axes[1].set_ylabel('count')
axes[1].set_title(f'{CELL} force magnitude distribution')
axes[1].legend()

plt.tight_layout()
fig.savefig(RUN_DIR / f'landscape_{CELL}.png', dpi=150)
plt.show()
model.train()

## 9. Cross-cell comparison dashboard

Run after completing multiple cells to see the full comparison.

In [ ]:
ALL_CELLS = sorted(RECIPES.keys())

dashboard = {}
for cell_name in ALL_CELLS:
    cell_dir = RESULTS_ROOT / cell_name / f'seed{SEED}'
    if not cell_dir.exists():
        dashboard[cell_name] = None
        continue
    log_file = cell_dir / 'training_log.jsonl'
    if not log_file.exists():
        dashboard[cell_name] = None
        continue
    evals = []
    with open(log_file) as f:
        for line in f:
            try:
                e = json.loads(line)
                if 'val_ppl' in e:
                    evals.append(e)
            except Exception:
                pass
    if not evals:
        dashboard[cell_name] = None
        continue
    best_ppl = min(e['val_ppl'] for e in evals)
    final_ppl = evals[-1]['val_ppl']
    last_step = evals[-1]['step']

    ls_files = sorted(cell_dir.glob(f'landscape_stats_*.json'))
    ls = json.loads(ls_files[-1].read_text()) if ls_files else None

    dashboard[cell_name] = {
        'desc': RECIPES[cell_name]['desc'],
        'best_ppl': best_ppl, 'final_ppl': final_ppl,
        'last_step': last_step, 'landscape': ls,
    }

print(f'{"Cell":<6} {"Description":<52} {"best PPL":>10} {"V range":>8} {"F max":>8} {"Bounded":>8}')
print('─' * 96)
print(f'{"":6} {"MLP baseline (8.95 PPL)":<52} {"8.95":>10} {"—":>8} {"—":>8} {"—":>8}')
print(f'{"":6} {"SQ3 A2 (10.36 PPL)":<52} {"10.36":>10} {"—":>8} {"—":>8} {"No":>8}')
print('─' * 96)

for cell_name in ALL_CELLS:
    r = dashboard[cell_name]
    desc = RECIPES[cell_name]['desc'][:50]
    if r is None:
        print(f'{cell_name:<6} {desc:<52} {"—":>10} {"—":>8} {"—":>8} {"—":>8}    (not run)')
    else:
        bounded = 'Yes' if (r['landscape'] or {}).get('bounded', False) else 'No'
        v_range = f"{r['landscape']['V_range']:.2f}" if r['landscape'] else '—'
        f_max = f"{r['landscape']['force_max']:.2f}" if r['landscape'] else '—'
        print(f'{cell_name:<6} {desc:<52} {r["best_ppl"]:>10.2f} {v_range:>8} {f_max:>8} {bounded:>8}')

## 10. Save summary

In [ ]:
summary_path = RUN_DIR / f'summary_{CELL}.md'
with open(summary_path, 'w') as f:
    f.write(f'# {CELL}: {recipe["desc"]}\n\n')
    f.write(f'| Setting | Value |\n')
    f.write(f'|---------|-------|\n')
    f.write(f'| V_theta kind | {recipe["v_theta_kind"]} |\n')
    if recipe['K_mix'] is not None:
        f.write(f'| K_mix | {recipe["K_mix"]} |\n')
    if recipe['sarf_n_anchors'] is not None:
        f.write(f'| SARF N_S | {recipe["sarf_n_anchors"]} |\n')
    if recipe['bg_quad_eps'] > 0:
        f.write(f'| bg_quad_eps | {recipe["bg_quad_eps"]} |\n')
    f.write(f'| xi_channels | {XI_CHANNELS} |\n')
    f.write(f'| lambda_V | {LAMBDA_V} |\n')
    f.write(f'| d | {D} |\n')
    f.write(f'| L | {L} |\n')
    f.write(f'| steps | {STEPS} |\n')
    f.write(f'| best PPL | {best_val_ppl:.2f} |\n')
    f.write(f'| V_theta params | {n_v_theta:,} |\n')
    f.write(f'| total params | {n_total:,} |\n')
    f.write(f'| Bounded V | {IS_GAUSSIAN} |\n')
    f.write(f'\n## Reference\n\n')
    f.write(f'- Fock-PARFLM v2.1 MLP baseline: **8.95 PPL**\n')
    f.write(f'- SQ3 K=8 (A2 winner): **10.36 PPL**\n')

print(f'Summary saved to {summary_path}')
print(f'\nFinal result: {CELL} = {best_val_ppl:.2f} PPL')
print(f'  Reference: MLP baseline = 8.95 PPL, SQ3 A2 = 10.36 PPL')